In [ ]:
"""
CarbonTatva AI — 24-Hour Ahead Electricity Price Forecasting
=============================================================
Three models trained separately for GDAM, DAM, RTM:
  Model 1: LightGBM
  Model 2: CNN-LSTM
  Model 3: Temporal Fusion Transformer (TFT)

Usage:
    pip install -r requirements.txt
    python train_models.py

Output:
    outputs/  — saved models, scalers, plots, metrics
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import json
from datetime import datetime

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_FILE  = 'final_merged_data2.xlsx'
MARKETS    = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY  = 96          # 96 × 15 min = 24 hours
RANDOM_SEED = 42

# Train/Val/Test split dates
TRAIN_END = '2024-12-31'
VAL_END   = '2025-09-30'
# Test = everything after VAL_END

np.random.seed(RANDOM_SEED)

# ══════════════════════════════════════════════════════════════════
# STEP 1 — LOAD & FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1: Loading and transforming data")
print("=" * 60)

sheets = pd.read_excel(DATA_FILE, sheet_name=None)


def build_features(df):
    """
    Full feature engineering pipeline.
    Input : raw sheet dataframe
    Output: enriched dataframe with all 13 model features
    """
    df = df.copy()

    # ── Datetime ──────────────────────────────────────────────
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    # ── Drop redundant/collinear columns ──────────────────────
    # apparent_temperature  → r=0.97 with temperature_2m  → DROP
    # sunshine_duration     → r=0.93 with shortwave_radiation → DROP
    # precipitation         → r=0.99 with rain              → DROP
    df = df.drop(columns=[
        'apparent_temperature (°C)',
        'sunshine_duration (s)',
        'precipitation (mm)'
    ], errors='ignore')

    # ── A) Temporal features ──────────────────────────────────
    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek   # 0=Mon
    df['month'] = df['datetime'].dt.month

    # Cyclical encodings (sin/cos so model knows 23h is close to 0h)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    # Holiday flag (Indian national holidays)
    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        # If holidays library not installed, use zeros
        df['holiday_flag'] = 0
        print("  [Note] pip install holidays for holiday_flag feature")

    # ── B) Autoregressive features ────────────────────────────
    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)    # same slot yesterday
    df['price_lag672'] = p.shift(672)   # same slot last week
    df['ma3d']  = p.shift(1).rolling(288).mean()   # 3-day moving avg
    df['ma7d']  = p.shift(1).rolling(672).mean()   # 7-day moving avg

    # ── C) Weather / supply-side features ─────────────────────
    # temperature quadratic (U-shaped demand effect)
    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2

    # solar radiation (strongest weather signal, r=−0.36)
    df['radiation'] = df['shortwave_radiation (W/m²)']

    # interaction: solar × hour_sin captures duck curve explicitly
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']

    # wind speed at 100m (turbine hub height proxy)
    df['wind_speed'] = df['wind_speed_100m (km/h)']

    # cloud cover
    df['cloud_cover'] = df['cloud_cover (%)']

    # humidity (monsoon proxy)
    df['humidity'] = df['relative_humidity_2m (%)']

    # rain (keep one precipitation proxy)
    df['rain'] = df['rain (mm)']

    # ── Log-price target (compresses spike tail) ──────────────
    df['log_price'] = np.log1p(df['Price (Rs./MWh)'])

    return df


# Build features for each market
market_dfs = {}
for mkt in MARKETS:
    print(f"  Building features for {mkt}...")
    market_dfs[mkt] = build_features(sheets[mkt])
    print(f"    Shape: {market_dfs[mkt].shape}  |  "
          f"Date range: {market_dfs[mkt]['datetime'].min().date()} → "
          f"{market_dfs[mkt]['datetime'].max().date()}")

# ── Feature lists used in each model ─────────────────────────────
# (P(t-1) and P(t-4) EXCLUDED — not available 24h ahead)
LGBM_FEATURES = [
    # Temporal
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    # Autoregressive
    'price_lag96', 'price_lag672',
    'ma3d', 'ma7d',
    # Weather
    'radiation', 'radiation_x_hour_sin',
    'temp', 'temp_sq',
    'wind_speed', 'cloud_cover',
    'humidity', 'rain',
]

# Features fed at each past time step into CNN-LSTM encoder
SEQ_PAST_FEATURES = [
    'log_price',                          # past price history
    'hour_sin', 'hour_cos',               # time position
    'month_sin', 'month_cos',
    'radiation', 'temp', 'wind_speed',    # past weather actuals
]

# Features known for the future 96 slots (forecasts available)
SEQ_FUTURE_FEATURES = [
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dow_sin', 'dow_cos',
    'is_weekend', 'holiday_flag',
    'price_lag96', 'price_lag672',        # same slot D-1 and W-1
    'ma3d', 'ma7d',
    'radiation', 'temp', 'temp_sq',
    'wind_speed', 'cloud_cover',
    'humidity',
]

print(f"\nLightGBM features    : {len(LGBM_FEATURES)}")
print(f"CNN-LSTM past feats  : {len(SEQ_PAST_FEATURES)}")
print(f"CNN-LSTM future feats: {len(SEQ_FUTURE_FEATURES)}")


# ══════════════════════════════════════════════════════════════════
# STEP 2 — TRAIN/VAL/TEST SPLIT + SCALING
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 2: Train/Val/Test split and scaling")
print("=" * 60)

from sklearn.preprocessing import StandardScaler


def split_and_scale(df, feature_cols):
    """
    Time-based split. NEVER shuffle — future must not leak into train.
    Returns split dataframes and fitted scaler.
    """
    df = df.dropna(subset=feature_cols + ['log_price']).copy()

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]

    print(f"    Train: {len(train):,} rows  |  "
          f"Val: {len(val):,}  |  Test: {len(test):,}")

    # Fit scaler on TRAIN only — never on val/test
    scaler = StandardScaler()
    train_scaled = train.copy()
    val_scaled   = val.copy()
    test_scaled  = test.copy()

    train_scaled[feature_cols] = scaler.fit_transform(train[feature_cols])
    val_scaled[feature_cols]   = scaler.transform(val[feature_cols])
    test_scaled[feature_cols]  = scaler.transform(test[feature_cols])

    return train_scaled, val_scaled, test_scaled, scaler, train, val, test


# ══════════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:30s}  MAE={mae:.1f}  RMSE={rmse:.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def plot_predictions(y_true, y_pred, title, path, n_days=3):
    """Plot actual vs predicted for the first n_days of test set."""
    n = n_days * SLOTS_DAY
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27')

    # Full test set
    axes[0].plot(y_true[:500], color='#aaa', lw=0.8, label='Actual', alpha=0.8)
    axes[0].plot(y_pred[:500], color='#e74c3c', lw=0.8, label='Predicted', alpha=0.8)
    axes[0].set_title(f'{title} — First 500 slots of Test Set',
                      color='white', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].tick_params(colors='#aaa')
    axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')

    # Zoomed: first 3 days
    axes[1].plot(y_true[:n], color='#aaa', lw=1.2, label='Actual')
    axes[1].plot(y_pred[:n], color='#27ae60', lw=1.2, label='Predicted', ls='--')
    axes[1].set_title(f'{title} — First {n_days} Days (zoomed)',
                      color='white', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].tick_params(colors='#aaa')
    axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[1].set_xlabel('Slot (15 min each)', color='#aaa')

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


STEP 1: Loading and transforming data
  Building features for GDAM...
    Shape: (144864, 36)  |  Date range: 2022-01-12 → 2026-02-28
  Building features for DAM...
    Shape: (144864, 36)  |  Date range: 2022-01-12 → 2026-02-28
  Building features for RTM...
    Shape: (144116, 36)  |  Date range: 2022-01-12 → 2026-02-28

LightGBM features    : 20
CNN-LSTM past feats  : 8
CNN-LSTM future feats: 18

STEP 2: Train/Val/Test split and scaling


In [ ]:
import os

# Create the outputs directory if it doesn't exist
if not os.path.exists('outputs'):
    os.makedirs('outputs')
    print("Created 'outputs/' directory.")

In [ ]:
all_metrics = {'lgbm': {}, 'cnnlstm': {}, 'tft': {}}

In [ ]:
# # STEP 1 — Uninstall old, install correct
# !pip uninstall pytorch-lightning -y -q
# !pip install pytorch-forecasting==1.1.1 lightning==2.1.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 61.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 whic

In [ ]:
import pytorch_forecasting, lightning.pytorch as pl
print(pytorch_forecasting.__version__)  # 1.1.1
print(pl.__version__)                   # 2.1.0

1.1.1
2.1.0


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODEL 3 — Temporal Fusion Transformer (TFT) — FULL TRAINING BLOCK
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MODEL 3: Temporal Fusion Transformer (TFT)")
print("=" * 60)

try:
    from pytorch_forecasting import (
        TemporalFusionTransformer, TimeSeriesDataSet, QuantileLoss
    )
    from pytorch_forecasting.data import GroupNormalizer
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import TensorBoardLogger
    TFT_AVAILABLE = True
    print(f"  [OK] pytorch-forecasting imported successfully")
    print(f"       lightning path: {pl.__file__}")
except ImportError as e:
    print(f"  [SKIP] Missing dependency: {e}")
    TFT_AVAILABLE = False

if TFT_AVAILABLE:
    import os
    os.makedirs('outputs', exist_ok=True)

    known_reals = [
        'hour_sin', 'hour_cos',
        'month_sin', 'month_cos',
        'dow_sin', 'dow_cos',
        'is_weekend', 'holiday_flag',
        'price_lag96', 'price_lag672',
        'ma3d', 'ma7d',
        'radiation', 'temp', 'temp_sq',
        'wind_speed', 'cloud_cover', 'humidity',
    ]

    def train_tft(mkt, df, lookback=SLOTS_DAY * 7,
                  horizon=SLOTS_DAY, max_epochs=50):

        print(f"\n  ── Training TFT for market: {mkt} ──")

        # ── 1. Data prep ───────────────────────────────────────
        df = df.copy()
        df = df.rename(columns={'Price (Rs./MWh)': 'price_target'})
        df = df.dropna(subset=known_reals + ['price_target'])
        df = df.reset_index(drop=True)
        df['time_idx'] = df.index.astype(int)
        df['group']    = mkt

        print(f"     Total rows after dropna: {len(df)}")

        train_df = df[df['datetime'] <= TRAIN_END].reset_index(drop=True)
        val_df   = df[(df['datetime'] > TRAIN_END) &
                      (df['datetime'] <= VAL_END)].reset_index(drop=True)
        test_df  = df[df['datetime'] > VAL_END].reset_index(drop=True)

        print(f"     Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

        if len(train_df) < lookback * 2 or len(val_df) < horizon:
            print(f"  [ERROR] Not enough rows for {mkt}. Skipping.")
            return None

        # ── 2. TimeSeriesDataSet ───────────────────────────────
        training_dataset = TimeSeriesDataSet(
            train_df,
            time_idx='time_idx',
            target='price_target',
            group_ids=['group'],
            min_encoder_length=lookback // 2,
            max_encoder_length=lookback,
            min_prediction_length=horizon,
            max_prediction_length=horizon,
            time_varying_known_reals=known_reals,
            time_varying_unknown_reals=['price_target'],
            target_normalizer=GroupNormalizer(
                groups=['group'], transformation='softplus'
            ),
            add_relative_time_idx=True,
            add_target_scales=True,
            add_encoder_length=True,
        )
        print(f"     TimeSeriesDataSet built. Samples: {len(training_dataset)}")

        val_dataset = TimeSeriesDataSet.from_dataset(
            training_dataset, val_df,
            predict=False, stop_randomization=True
        )

        train_loader = training_dataset.to_dataloader(
            train=True,  batch_size=64, num_workers=0, shuffle=True)
        val_loader   = val_dataset.to_dataloader(
            train=False, batch_size=64, num_workers=0)

        # ── 3. TFT model ───────────────────────────────────────
        tft = TemporalFusionTransformer.from_dataset(
            training_dataset,
            learning_rate=0.001,
            hidden_size=64,
            attention_head_size=4,
            dropout=0.1,
            hidden_continuous_size=16,
            loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
            log_interval=10,           # logs figures every 10 batches → needs TensorBoard
            reduce_on_plateau_patience=4,
        )
        print(f"     TFT parameters: {tft.size():,}")

        # ── 4. Callbacks ───────────────────────────────────────
        early_stop = EarlyStopping(
            monitor='val_loss', patience=8,
            verbose=True, mode='min'
        )
        checkpoint_cb = ModelCheckpoint(
            dirpath='outputs',
            filename=f'tft_{mkt}_best',
            monitor='val_loss',
            mode='min',
            save_top_k=1,
        )

        # ── 5. Trainer ─────────────────────────────────────────
        # TensorBoardLogger is REQUIRED — pytorch_forecasting calls
        # self.logger.experiment.add_figure() internally which is
        # a TensorBoard-only API. logger=False or CSVLogger will crash.
        tb_logger = TensorBoardLogger(
            save_dir='outputs/tb_logs',
            name=mkt
        )

        trainer = pl.Trainer(
            max_epochs=max_epochs,
            accelerator='auto',
            gradient_clip_val=0.1,
            callbacks=[early_stop, checkpoint_cb],
            enable_progress_bar=True,
            logger=tb_logger,           # ← TensorBoardLogger, NOT False
        )

        print(f"     Starting trainer.fit()...")
        trainer.fit(tft, train_dataloaders=train_loader,
                        val_dataloaders=val_loader)
        print(f"     Training complete. Best val_loss: {checkpoint_cb.best_model_score:.4f}")

        # ── 6. Load best checkpoint ────────────────────────────
        best_path = checkpoint_cb.best_model_path
        if not best_path:
            print("  [WARN] No checkpoint saved — using last epoch weights.")
            best_tft = tft
        else:
            best_tft = TemporalFusionTransformer.load_from_checkpoint(best_path)
            print(f"     Loaded best model from: {best_path}")

        # ── 7. Test set evaluation ─────────────────────────────
        test_dataset = TimeSeriesDataSet.from_dataset(
            training_dataset, test_df,
            predict=False, stop_randomization=True
        )
        test_loader = test_dataset.to_dataloader(
            train=False, batch_size=64, num_workers=0
        )

        print("     Running predictions on test set...")
        predictions = best_tft.predict(
            test_loader, return_y=True,
            trainer_kwargs=dict(
                accelerator='auto',
                logger=TensorBoardLogger(save_dir='outputs/tb_logs', name=f'{mkt}_test')
            )
        )

        y_pred = predictions.output[:, 1].numpy().flatten()   # P50
        y_true = predictions.y[0].numpy().flatten()

        print(f"\n  ── {mkt} Test Metrics ──")
        m = compute_metrics(y_true, y_pred, 'Test (P50)')
        all_metrics['tft'][mkt] = m

        # ── 8. Quantile plot ───────────────────────────────────
        p10 = predictions.output[:, 0].numpy().flatten()
        p50 = predictions.output[:, 1].numpy().flatten()
        p90 = predictions.output[:, 2].numpy().flatten()

        n = min(SLOTS_DAY * 5, len(p50))
        fig, ax = plt.subplots(figsize=(16, 6))
        fig.patch.set_facecolor('#0f1117')
        ax.set_facecolor('#1a1d27')
        ax.fill_between(range(n), p10[:n], p90[:n],
                        color='#2980b9', alpha=0.3, label='P10–P90 band')
        ax.plot(p50[:n], color='#2980b9', lw=1.5, label='P50 forecast')
        ax.plot(y_true[:n], color='white', lw=1, alpha=0.8, label='Actual')
        ax.set_title(f'TFT {mkt} — Probabilistic Forecast (first 5 test days)',
                     color='white', fontsize=12)
        ax.set_xlabel('Slot (15 min each)', color='#aaa')
        ax.set_ylabel('Price (Rs./MWh)', color='#aaa')
        ax.legend(fontsize=9)
        ax.tick_params(colors='#aaa')
        for spine in ax.spines.values():
            spine.set_edgecolor('#333')
        plt.tight_layout()
        plot_path = f'outputs/tft_pred_{mkt}.png'
        plt.savefig(plot_path, dpi=130, bbox_inches='tight', facecolor='#0f1117')
        plt.close()
        print(f"     Plot saved: {plot_path}")

        return best_tft

    # ── Run for all markets ────────────────────────────────────
    tft_models = {}
    all_metrics.setdefault('tft', {})

    for mkt in MARKETS:
        try:
            tft_models[mkt] = train_tft(mkt, market_dfs[mkt])
        except Exception as e:
            print(f"\n  [ERROR] TFT failed for {mkt}: {e}")
            import traceback; traceback.print_exc()
            tft_models[mkt] = None

    print("\n  TFT training complete for all markets.")


MODEL 3: Temporal Fusion Transformer (TFT)
  [OK] pytorch-forecasting imported successfully
       lightning path: /usr/local/lib/python3.12/dist-packages/lightning/pytorch/__init__.py

  ── Training TFT for market: GDAM ──
     Total rows after dropna: 144192
     Train: 103393 | Val: 26208 | Test: 14591
     TimeSeriesDataSet built. Samples: 103298


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: IPU available: False, using: 0 IPUs
INFO:lightning.pytorch.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs


     TFT parameters: 353,107
     Starting trainer.fit()...


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 0     
3  | prescalers                         | ModuleDict                      | 736   
4  | static_variable_selection          | VariableSelectionNetwork        | 9.4 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 70.5 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 66.6 K
7  | static_context_variable_selection  | GatedResidualNetwork            | 16.8 K
8  | static_context_initia

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


  [ERROR] TFT failed for GDAM: unsupported format string passed to NoneType.__format__

  ── Training TFT for market: DAM ──
     Total rows after dropna: 144192
     Train: 103393 | Val: 26208 | Test: 14591


Traceback (most recent call last):
  File "/tmp/ipykernel_1959/2804015167.py", line 214, in <cell line: 0>
    tft_models[mkt] = train_tft(mkt, market_dfs[mkt])
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1959/2804015167.py", line 143, in train_tft
    print(f"     Training complete. Best val_loss: {checkpoint_cb.best_model_score:.4f}")
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: unsupported format string passed to NoneType.__format__
